# Profiling experiment notebook

Goals: set up an experiment workflow, collect and visualize profiling results, and compare profile runs to document improvements.

Use this notebook to iterate: modify code → re-run profiles → re-run helper script → re-open notebook → compare runs.

## 1) Setup environment & imports
Install dev deps (once):
```bash
pip install -r requirements-dev.txt || pip install pandas matplotlib seaborn psutil pyinstrument snakeviz line_profiler
```
Then run the imports cell below.

In [ ]:
# Setup imports
import os, subprocess, json, time, glob
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pstats
import psutil
sns.set(style='whitegrid')

# Top-level paths
ROOT = Path('..').resolve() if Path('..').exists() else Path('.')
PROFILES_DIR = Path('..') / 'profiles' if (Path('..') / 'profiles').exists() else Path('profiles')
SUMMARY_DIR = PROFILES_DIR / 'summary'
print('Profiles dir:', PROFILES_DIR)
print('Summary dir:', SUMMARY_DIR)

## 2) Open project in VS Code / branch management
Commands to use in terminal:
```bash
git status
git checkout -b demo/experiment-xyz
code .
```
Use the VS Code source control panel to manage commits and the integrated terminal to run profiling commands.

## 3) Run baseline tests & lint
Record baseline test runtimes and linter output (example commands):
```bash
pytest -q --maxfail=1 --durations=10 | tee tests_baseline.log
flake8 . | tee flake8_baseline.log
black --check . | tee black_check.log
```

## 4) Create reproducible sandbox branch
Pattern: make small change, commit, profile, compare. Example:
```bash
git add -A && git commit -m "exp: tweak caching in intersections_with_damage"
git push -u origin HEAD
```

## 5) Instrument code for profiling (snippets)
Use `cProfile` for deterministic sampling and `pyinstrument` for hierarchical views. Example snippets:

In [ ]:
# cProfile snippet
import cProfile, pstats
pr = cProfile.Profile()
pr.enable()
# call target function here, e.g. run_main()
# pr.disable()
# pr.dump_stats('profiles/example.prof')

# pyinstrument snippet (if installed)
# from pyinstrument import Profiler
# p = Profiler()
# p.start()
# <call>
# p.stop()
# print(p.output_text(unicode=True, color=True))

## 6) Run and collect profiles (single-run)
Example command to run a single scenario and produce a .prof file and a simple metadata JSON:
```bash
python -m cProfile -o profiles/2_event1_`git rev-parse --short HEAD`_$(date +%s).prof scripts/2_intersection_analysis.py 30 1
```

## 7) Aggregate and store profiling artifacts
Run the helper to summarize .prof files into CSVs used by the notebook. If you haven't yet run it, do so now using the next cell.

In [ ]:
# Ensure summary CSVs exist (run the helper if needed)
SUMMARY_DIR = Path('profiles') / 'summary'
if not SUMMARY_DIR.exists() or len(list(SUMMARY_DIR.glob('*_top.csv'))) == 0:
    print('Summary CSVs missing; running tools/compare_profiles.py to generate them...')
    subprocess.run([sys.executable if 'sys' in globals() else 'python', 'tools/compare_profiles.py', '--profiles_dir', 'profiles', '--out_dir', 'profiles/summary'], check=False)
else:
    print('Found summary CSVs in', SUMMARY_DIR)

## 8) Visualize profiles (top functions)
Set the two profiles you want to compare below and run the plotting cell.

In [ ]:
# Config: pick two profile names (prefixes of .prof / _top.csv files)
PROFILE_A = '2_intersection_analysis_depth30_event1'
PROFILE_B = '2_intersection_analysis_depth30_event2'
TOP_N = 25

def load_top_csv(name):
    p = Path('profiles/summary') / f'{name}_top.csv'
    if not p.exists():
        raise FileNotFoundError(p)
    return pd.read_csv(p)

try:
    a = load_top_csv(PROFILE_A)
    b = load_top_csv(PROFILE_B)
except Exception as e:
    print('Error loading top CSVs:', e)
    a = b = pd.DataFrame()

if not a.empty and not b.empty:
    # align by function name and take top N union
    union_funcs = pd.Index(a['func']).union(b['func']).tolist()[:TOP_N]
    a_sub = a.set_index('func').reindex(union_funcs).fillna(0).reset_index()
    b_sub = b.set_index('func').reindex(union_funcs).fillna(0).reset_index()
    df_plot = pd.DataFrame({
        'func': union_funcs,
        PROFILE_A: a_sub['cumtime'].values,
        PROFILE_B: b_sub['cumtime'].values,
    })
    df_plot = df_plot.melt(id_vars='func', var_name='profile', value_name='cumtime')
    plt.figure(figsize=(10,8))
    sns.barplot(data=df_plot, x='cumtime', y='func', hue='profile')
    plt.title('Top functions cumulative time: {} vs {}'.format(PROFILE_A, PROFILE_B))
    plt.tight_layout()
    plt.show()
else:
    print('No data to plot; run tools/compare_profiles.py first.')

## 9) Automated experiment matrix (param sweeps)
You can define a list of parameter dictionaries and loop through them, running the target script under `cProfile` and saving outputs. Keep runs small and deterministic where possible.

In [ ]:
# Example experiment matrix (do not auto-run heavy jobs here)
experiments = [
    {'name':'base_30_e1','cmd':['python','-m','cProfile','-o','profiles/exp_base_30_e1.prof','scripts/2_intersection_analysis.py','30','1']},
    {'name':'base_30_e2','cmd':['python','-m','cProfile','-o','profiles/exp_base_30_e2.prof','scripts/2_intersection_analysis.py','30','2']},
]
print('Define experiments and run them from the terminal to avoid long notebook blocking.')

## 10) Compare profiles (baseline vs candidate)
This cell computes deltas between two aggregated top-cumulative times and lists functions with largest change.

In [ ]:
# Compare top functions and compute relative change
if not a.empty and not b.empty:
    merged = a.set_index('func')[['cumtime']].rename(columns={'cumtime':PROFILE_A}).join(
        b.set_index('func')[['cumtime']].rename(columns={'cumtime':PROFILE_B}), how='outer').fillna(0)
    merged['delta'] = merged[PROFILE_B] - merged[PROFILE_A]
    merged['pct_change'] = merged['delta'] / (merged[PROFILE_A].replace(0, pd.NA)) * 100
    merged_sorted = merged.reindex(merged['delta'].abs().sort_values(ascending=False).index)
    display(merged_sorted.head(30))
else:
    print('No data to compare.')

## 11) Document changes, metrics, and plots in this notebook
For every experiment, add a small markdown entry with: hypothesis, change summary, git commit hash, key plots, and short interpretation. Keep entries timestamped.

## 12) Reproducibility: environment capture and artifacts
Capture environment and artifacts as part of each experiment:
```bash
python --version > experiments/env_python.txt
pip freeze > experiments/requirements-dev.txt
```
Also include a small `experiment_manifest.json` listing commit, branch, and artifact paths.

## 13) Next actionable experiments checklist
- Cache `road_links` and `base_scenario_links` to avoid repeated disk reads (target: `scripts/2_intersection_analysis.py`).
- Reduce geometry precision / simplify before intersection (target: clip & simplify geometry).
- Parallelize raster intersections (process each raster in a separate process) and merge results.
- Measure memory usage (psutil) while running heavy steps to find allocations.
- Re-run profiles and use this notebook to compare improvements.